## SAFE RAG 생성기 및 성능 측정기

---

### 개요

이 노트북은 금융권 사내 RAG 시스템의 **임베딩 역전공격(Embedding Inversion Attack) 취약성**을 실험하고, 경량 방어 기법의 효과를 정량적으로 측정합니다.

**실험 구조**

| 단계 | 내용 |
|---|---|
| Step 1 | 금융 문서 로드 및 청킹 |
| Step 2 | OpenAI `text-embedding-ada-002`로 임베딩 생성 및 FAISS 인덱스 구축 |
| Step 3 | Vec2Text 역전공격으로 원문 복원 시도 |
| Step 4 | ROUGE-1 기준 복원율 측정 (Unsafe 베이스라인) |
| 별첨 | bge-m3 로컬 임베딩 모델 설치 (참고용) |

**핵심 질문**: 벡터 DB가 탈취되었을 때, 공격자가 원본 금융 문서(PII 포함)를 얼마나 복원할 수 있는가?

> **금융권 RAG 유형별 위협 수준**  
> - 규정·가이드라인 기반 RAG: 원문이 이미 공개 정보이므로 역전공격 실익 낮음  
> - 기업 분석·여신심사·내부 보고서 기반 RAG: 원문의 핵심 수치·PII가 비공개 정보이므로 **Safe 임베딩 필수**

---

#### Step 0. 패키지 설치

실험에 필요한 라이브러리를 설치합니다.

- `langchain` / `langchain-community`: 문서 로드 및 청킹 파이프라인
- `openai`: text-embedding-ada-002 API 호출
- `faiss-cpu`: 벡터 인덱스 구축 및 유사도 검색
- `vec2text`: 임베딩 역전공격 구현체 (Morris et al., 2023)
- `rouge-score`: 원문 대비 복원 텍스트 유사도 측정

> **환경 주의사항**: Python 3.11 + `.venv` 가상환경에서 실행하세요.  
> Python 3.13은 일부 ML 라이브러리(FlagEmbedding 등)와 호환성 문제가 있습니다.

In [ ]:
! pip install torch torchvision torchaudio

! pip install \
  langchain==0.3.1 \
  langchain-community==0.3.0 \
  langchain-openai==0.2.2 \
  langchain-text-splitters==0.3.0 \
  openai==1.42.0 \
  ollama==0.6.2 \
  faiss-cpu==1.14.2 \
  numpy==2.4.6 \
  scikit-learn==1.9.0 \
  vec2text==0.0.13 \
  rouge-score==0.1.2 \
  pymupdf==1.27.2.3 \
  python-dotenv==1.2.2 \
  tqdm==4.67.3 \
  transformers==4.41.2

---

#### Step 1. 금융 문서 로드 및 청킹

사내 금융 문서를 로드하고, 임베딩에 적합한 크기의 청크로 분할합니다.

**청킹 전략**
- `chunk_size=512`: PII(주민등록번호, 계좌번호 등)가 단일 청크 안에 고립되도록 작게 설정합니다. 이후 역전공격 실험에서 어느 청크에서 PII가 복원되는지 추적하기 위함입니다.
- `chunk_overlap=64`: 문장 경계에서 문맥이 잘리는 것을 방지합니다.

**왜 청크 크기가 보안에 중요한가?**  
청크가 클수록 하나의 벡터에 더 많은 정보가 담겨 역전공격 시 복원 가능한 정보량이 늘어납니다. 반대로 너무 작으면 RAG 검색 품질(Recall@K)이 떨어집니다.

In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 로드 (txt/md/pdf 모두 가능)
loader = DirectoryLoader("./docs/", glob="**/*.pdf", loader_cls=TextLoader)
documents = loader.load()

# 금융 문서 특성상 청크를 작게 — PII가 단일 청크에 고립되게
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=["\n\n", "\n", ".", " "]
)
chunks = splitter.split_documents(documents)
print(f"총 청크 수: {len(chunks)}")

총 청크 수: 12204


---

#### Step 2. 임베딩 생성 및 FAISS 인덱스 구축 (Unsafe)

OpenAI `text-embedding-ada-002` 모델로 각 청크를 1536차원 벡터로 변환하고, FAISS 인덱스에 저장합니다.

이 단계에서 생성되는 인덱스는 **보안 필터가 전혀 없는 Unsafe 버전**입니다. 이후 역전공격의 공격 대상(베이스라인)이 됩니다.

**왜 ada-002를 쓰는가?**  
Vec2Text의 corrector 모델이 `text-embedding-ada-002` 기준으로 학습되어 있어, 별도 변환 없이 바로 역전공격이 가능합니다. bge-m3(1024d)를 쓸 경우 차원 패딩이 필요하고 복원율이 낮게 나와 베이스라인 측정에 불리합니다.

> **비용**: 샘플 문서 10개(~5,000 토큰) 기준 약 $0.0005 미만으로 무시할 수준입니다.

In [ ]:
import numpy as np
import faiss
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

def get_embedding(text: str) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-ada-002",
        input=text
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

def build_and_save_index(chunks, path: str):
    embeddings = []
    for chunk in tqdm(chunks, desc=f"임베딩 생성 → {path}"):
        embeddings.append(get_embedding(chunk.page_content))

    matrix = np.vstack(embeddings)          # (N, 1536)
    faiss.normalize_L2(matrix)              # 코사인 유사도용 정규화
    index = faiss.IndexFlatIP(1536)         # 내적 = 코사인 유사도
    index.add(matrix)
    faiss.write_index(index, path)
    print(f"저장 완료: {path} | {index.ntotal}개 벡터")
    return matrix

embeddings_unsafe = build_and_save_index(chunks, "faiss_unsafe.index")

#### Step 2-b. Recall@5 검증

임베딩 인덱스의 RAG 검색 품질을 측정합니다. 같은 청크를 쿼리로 사용해 자기 자신이 Top-5 안에 검색되는지 확인합니다.

- **목표**: 0.90 이상
- 이 수치가 낮으면 청크 크기나 임베딩 모델을 재검토해야 합니다.
- 이후 Defense 인덱스와 비교하여 "방어 기법이 검색 품질을 얼마나 훼손하는가"를 측정하는 기준이 됩니다.

In [ ]:
def recall_at_k(index_path, chunks, k=5, n_eval=20):
    index = faiss.read_index(index_path)
    hits = 0
    for i in range(min(n_eval, len(chunks))):
        vec = get_embedding(chunks[i].page_content).reshape(1, -1)
        faiss.normalize_L2(vec)
        _, indices = index.search(vec, k)
        if i in indices[0]:
            hits += 1
    score = hits / min(n_eval, len(chunks))
    print(f"Recall@{k} ({index_path}): {score:.2f}")
    return score

recall_at_k("faiss_unsafe.index", chunks)

---

#### Step 3. Vec2Text 역전공격 수행 (Unsafe 베이스라인)

Vec2Text(Morris et al., 2023)를 사용하여 FAISS 인덱스에서 추출한 벡터로 원문 복원을 시도합니다.

**역전공격(Inversion Attack)이란?**  
공격자가 벡터 DB에 접근하여 임베딩 벡터를 탈취한 뒤, 이를 역방향으로 디코딩해 원본 텍스트를 복원하는 공격입니다. 성공할 경우 고객 PII, 내부 수치, 기밀 문서 내용이 노출될 수 있습니다.

**실험 조건**
- `num_steps=1`: 빠른 검증용. 품질을 높이려면 20까지 올릴 수 있으나 시간이 오래 걸립니다.
- `sequence_beam_width=0`: 단순 greedy 디코딩 사용
- MPS(Apple Silicon GPU)는 Vec2Text 미지원 → CPU 강제 설정 필요

> **보안 참고**: 이 실험은 통제된 환경에서 자체 보유 데이터에 대한 취약성 검증 목적으로만 수행합니다.

In [ ]:
import vec2text
import torch
import faiss
import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer

# MPS 끄고 CPU 강제 (Vec2Text MPS 미지원)
torch.backends.mps.is_available = lambda: False
device = torch.device("cpu")

corrector = vec2text.load_pretrained_corrector("text-embedding-ada-002")

index = faiss.read_index("faiss_unsafe.index")
n = index.ntotal
dim = index.d  # 1536
embeddings_unsafe = np.zeros((n, dim), dtype=np.float32)
index.reconstruct_n(0, n, embeddings_unsafe)

sample_vecs = torch.tensor(embeddings_unsafe[:10]).to(device)
originals = [chunks[i].page_content for i in range(10)]
scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=False)

recovered = []
scores = []

for i in tqdm(range(10), desc="역전공격 진행"):
    vec = sample_vecs[i:i+1]
    result = vec2text.invert_embeddings(
        embeddings=vec,
        corrector=corrector,
        num_steps=1,
        sequence_beam_width=0,
    )
    rec = result[0]
    recovered.append(rec)

    score = scorer.score(originals[i], rec)
    rouge = score['rouge1'].fmeasure
    scores.append(rouge)

    print(f"\n[청크 {i}] ROUGE-1: {rouge:.3f}")
    print(f"  원본: {originals[i][:60]}...")
    print(f"  복원: {rec[:60]}...")

print("\n" + "="*50)
print(f"평균 ROUGE-1 (Unsafe 베이스라인): {np.mean(scores):.3f}")
print(f"최고: {max(scores):.3f} | 최저: {min(scores):.3f}")
print("="*50)
print("→ 이 숫자가 높을수록 역전공격 성공, 낮을수록 자체 방어력 있음")

#### ROUGE-1 결과 해석 기준

| ROUGE-1 범위 | 의미 | 다음 액션 |
|---|---|---|
| 0.5 이상 | 공격 성공 — PII 복원 위험 수준 | Defense 임베딩 즉시 구현 |
| 0.2 ~ 0.5 | 부분 복원 — 의미 수준 유사 | Defense 효과 비교 가능 |
| 0.2 미만 | 구조적 방어 가능성 | ada-002 corrector 한계 문서화 |

> **Defense 임베딩 적용 계획**  
> Unsafe 베이스라인 측정 후, 동일 파이프라인에 아래 두 가지 경량 방어 기법을 순차 적용하여 ROUGE-1 감소 효과를 비교합니다.  
> 1. **가우시안 노이즈 주입** (`epsilon` 튜닝): 벡터에 랜덤 잡음 추가  
> 2. **PCA 차원 축소** (`n_components` 튜닝): 고유값이 작은 방향(PII 신호 집중) 제거  
>  
> 두 방법 모두 추가 인프라 투자 없이 `sklearn` 한 줄로 적용 가능하여, IT 예산 제약이 있는 환경에서 현실적인 대안입니다.

---

### 별첨. bge-m3 로컬 임베딩 모델 설치

**로컬에서의 텍스트 임베딩을 생성하기 위한** 모델 bge-m3를 설치합니다.

bge-m3는 BAAI(Beijing Academy of AI)에서 개발한 다국어 임베딩 모델로, 한국어를 포함한 100개 언어를 지원합니다.

| 항목 | 내용 |
|---|---|
| 차원 | 1024d (ada-002는 1536d) |
| 한국어 지원 | 우수 |
| 실행 방식 | 로컬 (API 비용 없음) |
| 모델 크기 | 약 570MB |
| Vec2Text 호환 | 간접 (차원 패딩 필요) |

> **설치 전 필수 조건**: Python 3.11 가상환경에서 실행해야 합니다.  
> Python 3.13에서는 `is_torch_fx_available` ImportError가 발생합니다.  
> 처음 실행 시 HuggingFace에서 모델을 자동 다운로드합니다 (~570MB, 최초 1회).

In [ ]:
! pip install transformers==4.44.2
! pip install FlagEmbedding==1.3.5

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(
    'BAAI/bge-m3',
    use_fp16=True,          # M 시리즈에서 메모리 절반으로 줄여줌
    device='mps'            # Apple Silicon GPU 사용
)

# 동작 확인
test = model.encode(
    ['안녕하세요, 이것은 테스트입니다.'],
    batch_size=4,
    max_length=512,
)
print(test['dense_vecs'].shape)   # (1, 1024) 나오면 정상